# Eiko Quickstart
Welcome to the Eiko interactive environment. This notebook allows you to run and test Eiko directly in your browser without installing anything locally.

**⚠️ Important: Enable a GPU**<br>
Before running the cells below, ensure you have a GPU attached:
1. Go to the top menu and select **Runtime > Change runtime type**.
2. Under "Hardware accelerator", select **T4 GPU** (or any other available GPU).
3. Click **Save**.

Run the code cell below to install Eiko on the remote machine.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemError("GPU not detected. Please follow the instructions above to enable a GPU.")

print(f"Success! Connected to GPU: {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})")

print("Downloading and installing Eiko. This may take a minute...")

# Clone the repository to access example scripts and data
!git clone https://github.com/sebftw/Eiko.git

# Install Eiko along with the example dependencies
!pip install -q ./Eiko[examples]

import eiko.eiko_torch

After sucessfully running the cell above, you should be able to use Eiko.

The code cell below shows an example of Eiko usage. You can Navigate to **File** > **Save a copy** to copy and modify this notebook.

In [ ]:
from eiko import eiko

# 1. Setup device and grid parameters.
device = torch.device("cuda")
N = 101
dx = 0.001  # Grid spacing in meters (1 mm)
c = 1540.0  # Speed of sound in m/s (uniform medium)

# 2. Create the slowness map (1/velocity) on the device.
f = torch.full((N, N), 1.0 / c, dtype=torch.float32, device=device)

# 3. Initialize the time-of-flight field.
# Unknown points are set to infinity
u_init = torch.full((N, N), float('inf'), dtype=torch.float32, device=device)

# Set a point source at the center of the grid to time = 0.
center_idx = N // 2
u_init[center_idx, center_idx] = 0.0

# 4. Compute the numerical solution.
u = eiko(u_init, f, dx=dx)

# Ensure plots display inline in the notebook
%matplotlib inline

# 5. Visualize the result.
import matplotlib.pyplot as plt
import torch

# Create physical coordinate axes in millimeters using dx
axis_mm = (torch.arange(N) - center_idx) * dx * 1000;

# Set up plot (equivalent to 'figure')
plt.figure(figsize=(6, 6))

# 'imagesc' equivalent with physical extents.
# 'extent' maps the data coordinates to the axes.
extent = [axis_mm[0], axis_mm[-1], axis_mm[0], axis_mm[-1]]
plt.imshow(u.cpu() * 1e6, extent=extent, origin='lower', cmap='viridis')

# 'axis image' equivalent (forces equal pixel aspect ratio)
plt.gca().set_aspect('equal', adjustable='box')

# Format axes and text size
plt.title('Time-of-Flight', fontsize=14, fontweight='bold')
plt.xlabel('x (mm)', fontsize=12, fontweight='bold')
plt.ylabel('y (mm)', fontsize=12, fontweight='bold')

# Format colorbar
cb = plt.colorbar()
cb.set_label(r'Time ($\mu$s)', fontsize=12)

# Display the plot
plt.show()